In [0]:
USE weather_openmeteo.gold;
--drop table if exists weather_openmeteo.gold.fact_weather;

-- Create Gold Fact Weather table if not exists with proper schema
CREATE TABLE IF NOT EXISTS weather_openmeteo.gold.fact_weather (
    weather_SK BIGINT GENERATED ALWAYS AS IDENTITY (START WITH 1 INCREMENT BY 1) NOT NULL, -- Surrogate Key
    date_FK BIGINT NOT NULL,
    date TIMESTAMP NOT NULL,
    city_FK BIGINT NOT NULL,
    temperature_2m_max DOUBLE,
    temperature_2m_min DOUBLE,
    daylight_duration DOUBLE,
    wind_speed_10m_max DOUBLE,
    avg_temperature_7d DOUBLE,
    alerts_temp BOOLEAN,
    alerts_wind BOOLEAN
)
USING DELTA;

-- Validate source data and prepare for upsert
WITH valid_fact AS (
    SELECT
        C.date_key,
        B.city_SK,
        A.date,
        A.temperature_2m_max,
        A.temperature_2m_min,
        A.daylight_duration,
        A.wind_speed_10m_max,
        A.avg_temperature_7d,
        A.alerts_temp,
        A.alerts_wind
    FROM weather_openmeteo.gold.weather_daily_kpis AS A
    INNER JOIN weather_openmeteo.gold.dim_city AS B
        ON A.latitude = B.latitude
        AND A.longitude = B.longitude
    INNER JOIN weather_openmeteo.gold.dim_date AS C
        ON CAST(DATE_FORMAT(A.date, 'yyyyMMdd') AS BIGINT) = C.date_key
    WHERE
        A.date IS NOT NULL
        AND B.city_SK IS NOT NULL
        AND (A.temperature_2m_max IS NOT NULL AND (A.temperature_2m_max BETWEEN -100 AND 100))
)
MERGE INTO weather_openmeteo.gold.fact_weather AS target
USING valid_fact AS source
ON target.date_FK = source.date_key AND target.city_FK = source.city_SK  
WHEN MATCHED THEN
  UPDATE SET
    target.temperature_2m_max = source.temperature_2m_max,
    target.temperature_2m_min = source.temperature_2m_min,
    target.daylight_duration = source.daylight_duration,
    target.wind_speed_10m_max = source.wind_speed_10m_max,
    target.avg_temperature_7d = source.avg_temperature_7d,
    target.alerts_temp = source.alerts_temp,
    target.alerts_wind = source.alerts_wind
WHEN NOT MATCHED THEN
  INSERT (
    date_FK, date, city_FK, temperature_2m_max, temperature_2m_min, daylight_duration, wind_speed_10m_max, avg_temperature_7d, alerts_temp, alerts_wind
  )
  VALUES (
    source.date_key, source.date, source.city_SK, source.temperature_2m_max, source.temperature_2m_min, source.daylight_duration, source.wind_speed_10m_max, source.avg_temperature_7d, source.alerts_temp, source.alerts_wind
  );

-- Display fact weather
SELECT * FROM weather_openmeteo.gold.fact_weather;